In [1]:

from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq

# Find the repository root
root = Path.cwd()
if not (root / "data/processed").exists():
    root = root.parent

data_path = root / "data/processed/patch_model_eligible_terrain_v2.parquet"
roles_path = root / "data/processed/patch_model_eligible_terrain_v2.roles.json"

assert data_path.exists(), "Dataset not found"
assert roles_path.exists(), "Feature roles not found"

# Load column-role definitions
with open(roles_path, "r") as f:
    roles = json.load(f)

# Inspect the dataset without reading any target values
parquet = pq.ParquetFile(data_path)

print("Dataset rows:", parquet.metadata.num_rows)
print("Dataset columns:", len(parquet.schema_arrow.names))
print("\nRoles JSON structure:")
print(json.dumps(roles, indent=2)[:3000])


Dataset rows: 111257
Dataset columns: 129

Roles JSON structure:
{
  "target": "nbr_5_mean",
  "predictors": [
    "nbr_pre_mean",
    "dominant_evt",
    "dominant_evt_fraction",
    "tree_cover_mean",
    "tree_height_mean",
    "elevation_mean",
    "slope_mean",
    "northness_mean",
    "eastness_mean",
    "fire_size_acres",
    "ignition_month",
    "mtbs_unburned_low_fraction",
    "mtbs_low_fraction",
    "mtbs_moderate_fraction",
    "mtbs_high_fraction",
    "mtbs_increased_greenness_fraction",
    "hist_prcp_mm",
    "hist_tmax_c",
    "hist_tmin_c",
    "hist_s1_vpd_kpa",
    "nbr_1_mean",
    "pre_s1_prcp_mm",
    "pre_s1_tmax_c",
    "pre_s1_tmin_c",
    "pre_s1_vpd_kpa",
    "pre_s1_prcp_anom_mm",
    "pre_s1_tmax_anom_c",
    "pre_s1_tmin_anom_c",
    "pre_s1_vpd_anom_kpa",
    "s1_prcp_anom_mm",
    "s1_tmax_anom_c",
    "s1_tmin_anom_c",
    "s1_vpd_anom_kpa",
    "nbr_drop_patch"
  ],
  "early_predictors": [
    "nbr_pre_mean",
    "dominant_evt",
    "dominant_evt_

In [2]:

# Approved inputs and target
features = roles["predictors"]
target = roles["target"]

assert len(features) == 34
assert target not in features

keys = ["fire_id", "patch_id", "split"]

# Load approved inputs for all splits
inputs = pd.read_parquet(
    data_path,
    columns=keys + features
)

# Load targets ONLY for training and validation
dev_targets = pd.read_parquet(
    data_path,
    columns=keys + [target],
    filters=[("split", "in", ["train", "validation"])]
)

# Combine inputs and labels for development data
dev = inputs.loc[
    inputs["split"].isin(["train", "validation"])
].merge(
    dev_targets,
    on=keys,
    validate="one_to_one"
)

train = dev.loc[dev["split"] == "train"].copy()
val = dev.loc[dev["split"] == "validation"].copy()

# Separate predictors (X) and target (y)
X_train, y_train = train[features], train[target]
X_val, y_val = val[features], val[target]

# Safety checks
assert len(train) == 61090
assert len(val) == 19561
assert inputs["split"].eq("test").sum() == 30606
assert train["fire_id"].nunique() == 78
assert val["fire_id"].nunique() == 36
assert set(train["fire_id"]).isdisjoint(val["fire_id"])
assert y_train.notna().all() and y_val.notna().all()

print("Training:", X_train.shape, "| Fires:", train["fire_id"].nunique())
print("Validation:", X_val.shape, "| Fires:", val["fire_id"].nunique())
print("Test inputs:", inputs["split"].eq("test").sum())
print("Target:", target)
print("Approved features:", len(features))


Training: (61090, 34) | Fires: 78
Validation: (19561, 34) | Fires: 36
Test inputs: 30606
Target: nbr_5_mean
Approved features: 34


In [4]:

import numpy as np
import sklearn
print(sklearn.__version__)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Learn the average NBR5 from TRAINING data only
train_mean = y_train.mean()

# 2. Predict that same value for every validation patch
mean_predictions = np.full(len(y_val), train_mean)

# 3. Calculate prediction errors
mae = mean_absolute_error(y_val, mean_predictions)
rmse = np.sqrt(mean_squared_error(y_val, mean_predictions))
r2 = r2_score(y_val, mean_predictions)

# 4. Give each validation wildfire equal importance
errors = pd.DataFrame({
    "fire_id": val["fire_id"].to_numpy(),
    "absolute_error": np.abs(y_val.to_numpy() - mean_predictions)
})

equal_fire_mae = (
    errors.groupby("fire_id")["absolute_error"]
    .mean()
    .mean()
)

print("MEAN BASELINE — VALIDATION RESULTS")
print(f"Training mean NBR5: {train_mean:.4f}")
print(f"Validation MAE:      {mae:.4f}")
print(f"Validation RMSE:     {rmse:.4f}")
print(f"Validation R²:       {r2:.4f}")
print(f"Equal-fire MAE:      {equal_fire_mae:.4f}")


1.9.1
MEAN BASELINE — VALIDATION RESULTS
Training mean NBR5: 0.3122
Validation MAE:      0.1624
Validation RMSE:     0.1932
Validation R²:       -0.0255
Equal-fire MAE:      0.1421


In [5]:

# Predict NBR5 using each patch's observed NBR1
persistence_predictions = X_val["nbr_1_mean"].to_numpy()

# Calculate validation prediction errors
persistence_mae = mean_absolute_error(
    y_val, persistence_predictions
)
persistence_rmse = np.sqrt(
    mean_squared_error(y_val, persistence_predictions)
)
persistence_r2 = r2_score(
    y_val, persistence_predictions
)

# Give each wildfire equal weight
persistence_errors = pd.DataFrame({
    "fire_id": val["fire_id"].to_numpy(),
    "absolute_error": np.abs(
        y_val.to_numpy() - persistence_predictions
    )
})

persistence_equal_fire_mae = (
    persistence_errors.groupby("fire_id")["absolute_error"]
    .mean()
    .mean()
)

print("PERSISTENCE BASELINE — VALIDATION RESULTS")
print(f"Validation MAE:     {persistence_mae:.4f}")
print(f"Validation RMSE:    {persistence_rmse:.4f}")
print(f"Validation R²:      {persistence_r2:.4f}")
print(f"Equal-fire MAE:     {persistence_equal_fire_mae:.4f}")

# Compare both baselines
comparison = pd.DataFrame({
    "Model": ["Training mean", "NBR1 persistence"],
    "MAE": [mae, persistence_mae],
    "RMSE": [rmse, persistence_rmse],
    "R²": [r2, persistence_r2],
    "Equal-fire MAE": [equal_fire_mae, persistence_equal_fire_mae]
})

print("\nBASELINE COMPARISON")
display(comparison.round(4))


PERSISTENCE BASELINE — VALIDATION RESULTS
Validation MAE:     0.1839
Validation RMSE:    0.2423
Validation R²:      -0.6139
Equal-fire MAE:     0.1585

BASELINE COMPARISON


,Model,MAE,RMSE,R²,Equal-fire MAE
0,Training mean,0.1624,0.1932,-0.0255,0.1421
1,NBR1 persistence,0.1839,0.2423,-0.6139,0.1585


In [7]:

import mlflow

# Store experiment records in a local SQLite database
db_path = root / "mlflow.db"

mlflow.set_tracking_uri(
    f"sqlite:///{db_path.as_posix()}"
)

mlflow.set_experiment("P1 - Wildfire Recovery - Phase 5")

print("MLflow version:", mlflow.__version__)
print("Tracking location:", mlflow.get_tracking_uri())
print("Experiment ready!")


2026/10/28 20:52:30 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/28 20:52:30 INFO mlflow.store.db.utils: Updating database tables
2026/10/28 20:52:35 INFO mlflow.tracking.fluent: Experiment with name 'P1 - Wildfire Recovery - Phase 5' does not exist. Creating a new experiment.


MLflow version: 3.17.2
Tracking location: sqlite:///c:/F copy/wildfire-recovery-forecasting/mlflow.db
Experiment ready!


In [8]:

# Shared information about our experiments
experiment_info = {
    "dataset": "patch_model_eligible_terrain_v2",
    "train_years": "2015-2017",
    "validation_years": "2018-2019",
    "train_fires": 78,
    "validation_fires": 36,
}

# Experiment 1: Training-mean baseline
with mlflow.start_run(run_name="baseline_training_mean"):
    mlflow.log_params({
        **experiment_info,
        "strategy": "predict_training_mean",
        "constant_prediction": float(train_mean),
    })

    mlflow.log_metrics({
        "val_mae": float(mae),
        "val_rmse": float(rmse),
        "val_r2": float(r2),
        "val_equal_fire_mae": float(equal_fire_mae),
    })

print("Logged training-mean baseline!")

# Experiment 2: NBR1 persistence baseline
with mlflow.start_run(run_name="baseline_nbr1_persistence"):
    mlflow.log_params({
        **experiment_info,
        "strategy": "predict_nbr1",
        "predictor": "nbr_1_mean",
    })

    mlflow.log_metrics({
        "val_mae": float(persistence_mae),
        "val_rmse": float(persistence_rmse),
        "val_r2": float(persistence_r2),
        "val_equal_fire_mae": float(persistence_equal_fire_mae),
    })

print("Logged NBR1 persistence baseline!")

# Retrieve both experiments from MLflow
runs = mlflow.search_runs(
    experiment_names=["P1 - Wildfire Recovery - Phase 5"]
)

display(
    runs[
        [
            "tags.mlflow.runName",
            "metrics.val_mae",
            "metrics.val_rmse",
            "metrics.val_r2",
            "metrics.val_equal_fire_mae",
        ]
    ].round(4)
)


Logged training-mean baseline!
Logged NBR1 persistence baseline!


,tags.mlflow.runName,metrics.val_mae,metrics.val_rmse,metrics.val_r2,metrics.val_equal_fire_mae
0,baseline_nbr1_persistence,0.1839,0.2423,-0.6139,0.1585
1,baseline_training_mean,0.1624,0.1932,-0.0255,0.1421


In [9]:
from wildfire_recovery_forecasting.evaluation import evaluate_model

# Check that the function reproduces our original results
mean_scores = evaluate_model(
    y_val, mean_predictions, val["fire_id"]
)

persistence_scores = evaluate_model(
    y_val, persistence_predictions, val["fire_id"]
)

assert np.isclose(mean_scores["mae"], mae)
assert np.isclose(mean_scores["equal_fire_mae"], equal_fire_mae)
assert np.isclose(persistence_scores["mae"], persistence_mae)
assert np.isclose(
    persistence_scores["equal_fire_mae"],
    persistence_equal_fire_mae
)

display(pd.DataFrame({
    "Training mean": mean_scores,
    "NBR1 persistence": persistence_scores
}).T.round(4))

print("Reusable evaluation function verified!")


,mae,rmse,r2,equal_fire_mae
Training mean,0.1624,0.1932,-0.0255,0.1421
NBR1 persistence,0.1839,0.2423,-0.6139,0.1585


Reusable evaluation function verified!
